# <center> Семинар 16. Кросс-доменные последовательные рекомендации: retail + reviews </center>

## Идея семинара

Проверяем вопрос: **помогают ли review-события улучшить рекомендации retail-товаров**.

Важно: `reviews` имеют **другую схему**, чем `retail`. Поэтому мы не сводим всё к одному `item_id`, а строим **разные event encoder-ы**:

- `retail` encoder: `item_id + action_type + subdomain + os`
- `reviews` encoder: `brand_id + rating + text_embedding`

Дальше оба типа событий переводятся в общий embedding space, и Transformer работает уже с гетерогенной последовательностью событий пользователя.


## План

1. Проверить схемы `retail` и `reviews`.
2. Сэмплировать активных пользователей по retail-событиям.
3. Построить единую временную последовательность событий.
4. Подготовить примеры для задачи next-item prediction в retail.
5. Реализовать модель с отдельными encoder-ами для retail и reviews.
6. Сравнить `retail_only` против `retail_plus_reviews`.


In [ ]:
# Если окружение пустое, можно раскомментировать:
# !pip install -q polars pyarrow numpy pandas torch tqdm


In [87]:
from __future__ import annotations

import math
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

pl.Config.set_tbl_rows(10)
pl.Config.set_tbl_cols(12)


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)

# DEVICE = (
#     "cuda" if torch.cuda.is_available() else
#     "mps" if torch.backends.mps.is_available() else
#     "cpu"
# )
DEVICE = 'cpu'


In [88]:
DATASET_ROOT = Path("dataset")
DATA_ROOT = DATASET_ROOT / "small"

TARGET_DOMAIN = "retail"
AUX_DOMAINS = ["reviews"]
SELECTED_DOMAINS = [TARGET_DOMAIN, *AUX_DOMAINS]

N_USERS = 3000
USE_LAST_DAYS = 50
MIN_RETAIL_EVENTS_PER_USER = 5
MAX_EVENTS_PER_USER = 100

TOP_TARGET_ITEMS = 2500
MAX_SEQ_LEN = 64

MIN_CAT_FREQ = 3

BATCH_SIZE = 256
EMBED_DIM = 128
N_HEADS = 4
N_LAYERS = 2
DROPOUT = 0.1
LR = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS = 4

REVIEW_EMBED_DIM = 312
SELECTED_DOMAINS


['retail', 'reviews']

## 1. Проверка структуры локальных данных


In [89]:
def list_event_files(domain: str) -> list[Path]:
    domain_dir = DATA_ROOT / domain / "events"
    if not domain_dir.exists():
        return []
    return sorted(domain_dir.glob("*.pq"))


print("DATASET_ROOT:", DATASET_ROOT.resolve())
print("DATA_ROOT exists:", DATA_ROOT.exists())
for domain in SELECTED_DOMAINS:
    files = list_event_files(domain)
    print(domain, len(files))


DATASET_ROOT: /Users/o.a.lashinin/Downloads/RecSys_course/week16/dataset
DATA_ROOT exists: True
retail 50
reviews 50


In [90]:
for domain in SELECTED_DOMAINS:
    files = list_event_files(domain)
    assert files, f"No files found for {domain} in {DATA_ROOT / domain / 'events'}"
    schema = pl.scan_parquet(str(files[-1])).collect_schema()
    print(f"\n[{domain}] example file: {files[-1].name}")
    for col, dtype in schema.items():
        print(f"  {col}: {dtype}")



[retail] example file: 01305.pq
  timestamp: Duration(time_unit='us')
  user_id: UInt64
  item_id: String
  subdomain: String
  action_type: String
  os: String

[reviews] example file: 01305.pq
  timestamp: Duration(time_unit='us')
  user_id: UInt64
  brand_id: UInt64
  rating: UInt8
  text: String
  embedding: Array(Float32, shape=(312,))


Схемы отличаются:

- `retail`: `item_id`, `subdomain`, `action_type`, `os`
- `reviews`: `brand_id`, `rating`, `text`, `embedding`

Поэтому reviews нельзя запихнуть в тот же encoder, что retail. Ниже мы делаем для них отдельный путь обработки.


## 2. Сэмплируем активных retail-пользователей


In [91]:
retail_files = list_event_files("retail")[-USE_LAST_DAYS:]
review_files = list_event_files("reviews")[-USE_LAST_DAYS:]

retail_user_stats = (
    pl.scan_parquet([str(p) for p in retail_files])
    .select(["user_id", "item_id", "timestamp"])
    .group_by("user_id")
    .agg(pl.len().alias("n_events"))
    .filter(pl.col("n_events") >= MIN_RETAIL_EVENTS_PER_USER)
    .collect()
)

review_users = (
    pl.scan_parquet([str(p) for p in review_files])
    .select([pl.col("user_id").cast(pl.UInt64)])
    .drop_nulls()
    .unique()
    .collect()
)

retail_active_users = set(retail_user_stats["user_id"].to_list())
review_user_set = set(review_users["user_id"].to_list())
eligible_users = sorted(retail_active_users & review_user_set)

print("Retail-active users:", len(retail_active_users))
print("Users with >=1 review:", len(review_user_set))
print("Eligible users with retail + at least one review:", len(eligible_users))

retail_user_stats.filter(pl.col("user_id").is_in(eligible_users)).head()


Retail-active users: 49020
Users with >=1 review: 11602
Eligible users with retail + at least one review: 700


user_id,n_events
u64,u32
33543625,120
41817391,45
2253401,648
87732737,25
45978917,6


In [92]:
candidate_users = eligible_users
sample_size = min(N_USERS, len(candidate_users))
selected_users = sorted(random.sample(candidate_users, sample_size))
print("Candidate users after review filter:", len(candidate_users))
print("Selected users:", len(selected_users))


Candidate users after review filter: 700
Selected users: 700


## 3. Загружаем и нормализуем события

Мы приведем события разных доменов к общей структуре, но **без потери модальности**: retail-поля и reviews-поля будут храниться отдельно.


In [93]:
def load_retail_events(users: list[int], use_last_days: int = USE_LAST_DAYS) -> list[dict]:
    files = list_event_files("retail")[-use_last_days:]
    print(f"Loading retail events from {len(files)} files for {len(users)} users")
    df = (
        pl.scan_parquet([str(p) for p in files])
        .filter(pl.col("user_id").is_in(users))
        .select([
            pl.col("timestamp").cast(pl.Int64).alias("timestamp"),
            pl.col("user_id").cast(pl.UInt64).alias("user_id"),
            pl.col("item_id").cast(pl.String).alias("item_id"),
            pl.col("subdomain").cast(pl.String).fill_null("unknown").alias("subdomain"),
            pl.col("action_type").cast(pl.String).fill_null("unknown").alias("action_type"),
            pl.col("os").cast(pl.String).fill_null("unknown").alias("os"),
        ])
        .drop_nulls(subset=["timestamp", "user_id", "item_id"])
        .collect()
        .sort(["user_id", "timestamp"])
    )

    print("Retail frame shape:", df.shape)
    rows = df.to_dicts()
    print("Converting retail frame to python events:", len(rows))
    events = []
    for row in tqdm(rows, desc="retail rows -> events"):
        events.append({
            "user_id": row["user_id"],
            "timestamp": row["timestamp"],
            "domain": "retail",
            "retail_item_id": row["item_id"],
            "retail_action_type": row["action_type"],
            "retail_subdomain": row["subdomain"],
            "retail_os": row["os"],
            "review_brand_id": None,
            "review_rating": None,
            "review_embedding": None,
            "target_item": row["item_id"],
        })
    print("Retail events prepared:", len(events))
    return events


def load_review_events(users: list[int], use_last_days: int = USE_LAST_DAYS) -> list[dict]:
    files = list_event_files("reviews")[-use_last_days:]
    print(f"Loading review events from {len(files)} files for {len(users)} users")
    df = (
        pl.scan_parquet([str(p) for p in files])
        .filter(pl.col("user_id").is_in(users))
        .select([
            pl.col("timestamp").cast(pl.Int64).alias("timestamp"),
            pl.col("user_id").cast(pl.UInt64).alias("user_id"),
            pl.col("brand_id").cast(pl.String).alias("brand_id"),
            pl.col("rating").cast(pl.Int64).alias("rating"),
            pl.col("embedding").alias("embedding"),
        ])
        .drop_nulls(subset=["timestamp", "user_id", "brand_id", "rating"])
        .collect()
        .sort(["user_id", "timestamp"])
    )

    print("Review frame shape:", df.shape)
    rows = df.to_dicts()
    print("Converting review frame to python events:", len(rows))
    zero_emb = [0.0] * REVIEW_EMBED_DIM
    events = []
    for row in tqdm(rows, desc="review rows -> events"):
        emb = row["embedding"]
        if emb is None:
            emb = zero_emb
        else:
            emb = list(emb)
            if len(emb) != REVIEW_EMBED_DIM:
                emb = (emb + zero_emb)[:REVIEW_EMBED_DIM]

        events.append({
            "user_id": row["user_id"],
            "timestamp": row["timestamp"],
            "domain": "reviews",
            "retail_item_id": None,
            "retail_action_type": None,
            "retail_subdomain": None,
            "retail_os": None,
            "review_brand_id": row["brand_id"],
            "review_rating": int(row["rating"]),
            "review_embedding": emb,
            "target_item": None,
        })
    print("Review events prepared:", len(events))
    return events


def trim_events_per_user(events: list[dict], max_events_per_user: int = MAX_EVENTS_PER_USER) -> list[dict]:
    print(f"Trimming to at most {max_events_per_user} events per user")
    by_user: dict[int, list[dict]] = {}
    for event in tqdm(events, desc="trim: group events by user"):
        by_user.setdefault(event["user_id"], []).append(event)

    trimmed = []
    total_before = 0
    total_after = 0
    for idx, (user_id, seq) in enumerate(tqdm(by_user.items(), total=len(by_user), desc="trim: keep last events"), start=1):
        seq = sorted(seq, key=lambda x: (x["timestamp"], x["domain"]))
        kept = seq[-max_events_per_user:]
        trimmed.extend(kept)
        total_before += len(seq)
        total_after += len(kept)
        if idx % 500 == 0:
            print(f"trimmed_users={idx} events_before={total_before} events_after={total_after}")

    print(f"Trimmed events from {total_before} to {total_after}")
    return sorted(trimmed, key=lambda x: (x["user_id"], x["timestamp"], x["domain"]))


retail_events = load_retail_events(selected_users)
review_events = load_review_events(selected_users)
events = sorted(retail_events + review_events, key=lambda x: (x["user_id"], x["timestamp"], x["domain"]))
events = trim_events_per_user(events, MAX_EVENTS_PER_USER)

print("retail events:", len(retail_events))
print("review events:", len(review_events))
print("all events after trim:", len(events))


Loading retail events from 50 files for 700 users
Retail frame shape: (1337105, 6)
Converting retail frame to python events: 1337105


retail rows -> events:   0%|          | 0/1337105 [00:00<?, ?it/s]

Retail events prepared: 1337105
Loading review events from 50 files for 700 users
Review frame shape: (2630, 5)
Converting review frame to python events: 2630


review rows -> events:   0%|          | 0/2630 [00:00<?, ?it/s]

Review events prepared: 2630
Trimming to at most 100 events per user


trim: group events by user:   0%|          | 0/1339735 [00:00<?, ?it/s]

trim: keep last events:   0%|          | 0/700 [00:00<?, ?it/s]

trimmed_users=500 events_before=976397 events_after=36121
Trimmed events from 1339735 to 50252
retail events: 1337105
review events: 2630
all events after trim: 50252


In [106]:
pd.Series([e["domain"] for e in events]).value_counts()


retail     48612
reviews     1640
Name: count, dtype: int64

In [109]:
events[1]

{'user_id': 1098,
 'timestamp': 108626621842004,
 'domain': 'retail',
 'retail_item_id': 'fmcg_907922',
 'retail_action_type': 'order',
 'retail_subdomain': 'other',
 'retail_os': 'android',
 'review_brand_id': None,
 'review_rating': None,
 'review_embedding': None,
 'target_item': 'fmcg_907922'}

## 4. Ограничиваем target space retail-товаров


In [95]:
target_counter = Counter(e["target_item"] for e in retail_events if e["target_item"] is not None)
target_item_set = {item for item, _ in target_counter.most_common(TOP_TARGET_ITEMS)}
print("Target items:", len(target_item_set))


Target items: 2500


## 5. Строим примеры для задачи next retail item

Target всегда берется из `retail`. В `retail_plus_reviews` в историю просто добавляются review-события пользователя до target-момента.


In [96]:
def build_examples(
    events: list[dict],
    history_domains: list[str],
    target_item_set: set[str],
    max_seq_len: int = MAX_SEQ_LEN,
) -> tuple[list[dict], list[dict], list[dict]]:
    print(f"Building examples for domains={history_domains}")
    print("Total raw events:", len(events))
    by_user: dict[int, list[dict]] = {}
    for event in tqdm(events, desc="group events by user"):
        by_user.setdefault(event["user_id"], []).append(event)
    print("Users with sequences:", len(by_user))

    train_examples = []
    valid_examples = []
    test_examples = []

    for idx, (user_id, seq) in enumerate(tqdm(by_user.items(), total=len(by_user), desc=f"build_examples:{'+'.join(history_domains)}"), start=1):
        filtered_seq = [x for x in seq if x["domain"] in history_domains]
        retail_positions = [
            idx for idx, row in enumerate(filtered_seq)
            if row["domain"] == "retail" and row["target_item"] in target_item_set
        ]

        if len(retail_positions) < 3:
            continue

        split_positions = {
            "train": retail_positions[:-2],
            "valid": [retail_positions[-2]],
            "test": [retail_positions[-1]],
        }

        for split_name, positions in split_positions.items():
            for pos in positions:
                history = filtered_seq[max(0, pos - max_seq_len):pos]
                if not history:
                    continue

                target_row = filtered_seq[pos]
                example = {
                    "user_id": user_id,
                    "history_events": history,
                    "target_item": target_row["target_item"],
                    "target_ts": target_row["timestamp"],
                }

                if split_name == "train":
                    train_examples.append(example)
                elif split_name == "valid":
                    valid_examples.append(example)
                else:
                    test_examples.append(example)

        if idx % 500 == 0:
            print(
                f"processed_users={idx} train={len(train_examples)} valid={len(valid_examples)} test={len(test_examples)}"
            )

    print("Finished build_examples")
    print("train / valid / test:", len(train_examples), len(valid_examples), len(test_examples))
    return train_examples, valid_examples, test_examples


## 6. Строим vocabulary для разных модальностей


In [97]:
def make_vocab(counter: Counter, min_freq: int = MIN_CAT_FREQ) -> dict[str, int]:
    vocab = {"[PAD]": 0, "[UNK]": 1}
    for key, freq in counter.items():
        if freq >= min_freq:
            vocab[key] = len(vocab)
    return vocab


def build_feature_vocabs(train_examples: list[dict]):
    retail_item_counter = Counter()
    retail_action_counter = Counter()
    retail_subdomain_counter = Counter()
    retail_os_counter = Counter()
    review_brand_counter = Counter()
    target_counter = Counter()

    for ex in train_examples:
        target_counter.update([ex["target_item"]])
        for ev in ex["history_events"]:
            if ev["domain"] == "retail":
                retail_item_counter.update([ev["retail_item_id"]])
                retail_action_counter.update([ev["retail_action_type"]])
                retail_subdomain_counter.update([ev["retail_subdomain"]])
                retail_os_counter.update([ev["retail_os"]])
            elif ev["domain"] == "reviews":
                review_brand_counter.update([ev["review_brand_id"]])

    vocabs = {
        "retail_item": make_vocab(retail_item_counter),
        "retail_action": make_vocab(retail_action_counter),
        "retail_subdomain": make_vocab(retail_subdomain_counter),
        "retail_os": make_vocab(retail_os_counter),
        "review_brand": make_vocab(review_brand_counter),
        "target_item": {item: idx for idx, item in enumerate(target_counter.keys())},
    }
    return vocabs


## 7. Кодируем примеры для модели


In [98]:
EVENT_TYPE_PAD = 0
EVENT_TYPE_RETAIL = 1
EVENT_TYPE_REVIEW = 2


def encode_examples(examples: list[dict], vocabs: dict[str, dict[str, int]], split_name: str = "unknown") -> list[dict]:
    print(f"Encoding {split_name} examples: {len(examples)}")
    encoded = []
    zero_emb = [0.0] * REVIEW_EMBED_DIM

    for idx, ex in enumerate(tqdm(examples, desc=f"encode:{split_name}"), start=1):
        if ex["target_item"] not in vocabs["target_item"]:
            continue

        event_type_ids = []
        retail_item_ids = []
        retail_action_ids = []
        retail_subdomain_ids = []
        retail_os_ids = []
        review_brand_ids = []
        review_rating_ids = []
        review_embedding_vectors = []

        for ev in ex["history_events"][-MAX_SEQ_LEN:]:
            if ev["domain"] == "retail":
                event_type_ids.append(EVENT_TYPE_RETAIL)
                retail_item_ids.append(vocabs["retail_item"].get(ev["retail_item_id"], 1))
                retail_action_ids.append(vocabs["retail_action"].get(ev["retail_action_type"], 1))
                retail_subdomain_ids.append(vocabs["retail_subdomain"].get(ev["retail_subdomain"], 1))
                retail_os_ids.append(vocabs["retail_os"].get(ev["retail_os"], 1))
                review_brand_ids.append(0)
                review_rating_ids.append(0)
                review_embedding_vectors.append(zero_emb)
            else:
                event_type_ids.append(EVENT_TYPE_REVIEW)
                retail_item_ids.append(0)
                retail_action_ids.append(0)
                retail_subdomain_ids.append(0)
                retail_os_ids.append(0)
                review_brand_ids.append(vocabs["review_brand"].get(ev["review_brand_id"], 1))
                review_rating_ids.append(int(ev["review_rating"]))
                review_embedding_vectors.append(ev["review_embedding"] if ev["review_embedding"] is not None else zero_emb)

        encoded.append({
            "event_type_ids": event_type_ids,
            "retail_item_ids": retail_item_ids,
            "retail_action_ids": retail_action_ids,
            "retail_subdomain_ids": retail_subdomain_ids,
            "retail_os_ids": retail_os_ids,
            "review_brand_ids": review_brand_ids,
            "review_rating_ids": review_rating_ids,
            "review_embedding_vectors": review_embedding_vectors,
            "target": vocabs["target_item"][ex["target_item"]],
            "target_item": ex["target_item"],
        })

        if idx % 5000 == 0:
            print(f"encoded {split_name}: {idx} / {len(examples)}")

    print(f"Finished encoding {split_name}: {len(encoded)} kept examples")
    return encoded


## 8. Dataset и collate


In [99]:
class SeqDataset(Dataset):
    def __init__(self, examples: list[dict]):
        self.examples = examples

    def __len__(self) -> int:
        return len(self.examples)

    def __getitem__(self, idx: int) -> dict:
        return self.examples[idx]


def collate_batch(batch: list[dict]) -> dict[str, torch.Tensor]:
    max_len = max(len(x["event_type_ids"]) for x in batch)

    event_type_ids = []
    retail_item_ids = []
    retail_action_ids = []
    retail_subdomain_ids = []
    retail_os_ids = []
    review_brand_ids = []
    review_rating_ids = []
    review_embedding_vectors = []
    attention_mask = []
    targets = []

    zero_emb = [0.0] * REVIEW_EMBED_DIM

    for ex in batch:
        seq_len = len(ex["event_type_ids"])
        pad_len = max_len - seq_len

        event_type_ids.append(ex["event_type_ids"] + [EVENT_TYPE_PAD] * pad_len)
        retail_item_ids.append(ex["retail_item_ids"] + [0] * pad_len)
        retail_action_ids.append(ex["retail_action_ids"] + [0] * pad_len)
        retail_subdomain_ids.append(ex["retail_subdomain_ids"] + [0] * pad_len)
        retail_os_ids.append(ex["retail_os_ids"] + [0] * pad_len)
        review_brand_ids.append(ex["review_brand_ids"] + [0] * pad_len)
        review_rating_ids.append(ex["review_rating_ids"] + [0] * pad_len)
        review_embedding_vectors.append(ex["review_embedding_vectors"] + [zero_emb] * pad_len)
        attention_mask.append([1] * seq_len + [0] * pad_len)
        targets.append(ex["target"])

    return {
        "event_type_ids": torch.tensor(event_type_ids, dtype=torch.long),
        "retail_item_ids": torch.tensor(retail_item_ids, dtype=torch.long),
        "retail_action_ids": torch.tensor(retail_action_ids, dtype=torch.long),
        "retail_subdomain_ids": torch.tensor(retail_subdomain_ids, dtype=torch.long),
        "retail_os_ids": torch.tensor(retail_os_ids, dtype=torch.long),
        "review_brand_ids": torch.tensor(review_brand_ids, dtype=torch.long),
        "review_rating_ids": torch.tensor(review_rating_ids, dtype=torch.long),
        "review_embedding_vectors": torch.tensor(review_embedding_vectors, dtype=torch.float32),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.bool),
        "targets": torch.tensor(targets, dtype=torch.long),
    }


## 9. Модель с отдельными encoder-ами для retail и reviews


In [100]:
class HeterogeneousSeqTransformer(nn.Module):
    def __init__(
        self,
        n_retail_items: int,
        n_retail_actions: int,
        n_retail_subdomains: int,
        n_retail_os: int,
        n_review_brands: int,
        n_targets: int,
        max_seq_len: int = MAX_SEQ_LEN,
        embed_dim: int = EMBED_DIM,
        n_heads: int = N_HEADS,
        n_layers: int = N_LAYERS,
        dropout: float = DROPOUT,
    ):
        super().__init__()

        self.event_type_emb = nn.Embedding(3, embed_dim)
        self.pos_emb = nn.Embedding(max_seq_len, embed_dim)

        self.retail_item_emb = nn.Embedding(n_retail_items, embed_dim, padding_idx=0)
        self.retail_action_emb = nn.Embedding(n_retail_actions, embed_dim, padding_idx=0)
        self.retail_subdomain_emb = nn.Embedding(n_retail_subdomains, embed_dim, padding_idx=0)
        self.retail_os_emb = nn.Embedding(n_retail_os, embed_dim, padding_idx=0)

        self.review_brand_emb = nn.Embedding(n_review_brands, embed_dim, padding_idx=0)
        self.review_rating_emb = nn.Embedding(6, embed_dim, padding_idx=0)
        self.review_text_proj = nn.Linear(REVIEW_EMBED_DIM, embed_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=n_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(embed_dim, n_targets)

    def forward(
        self,
        event_type_ids: torch.Tensor,
        retail_item_ids: torch.Tensor,
        retail_action_ids: torch.Tensor,
        retail_subdomain_ids: torch.Tensor,
        retail_os_ids: torch.Tensor,
        review_brand_ids: torch.Tensor,
        review_rating_ids: torch.Tensor,
        review_embedding_vectors: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> torch.Tensor:
        batch_size, seq_len = event_type_ids.shape
        positions = torch.arange(seq_len, device=event_type_ids.device).unsqueeze(0).expand(batch_size, -1)

        retail_vec = (
            self.retail_item_emb(retail_item_ids)
            + self.retail_action_emb(retail_action_ids)
            + self.retail_subdomain_emb(retail_subdomain_ids)
            + self.retail_os_emb(retail_os_ids)
        )

        review_vec = (
            self.review_brand_emb(review_brand_ids)
            + self.review_rating_emb(review_rating_ids.clamp(min=0, max=5))
            + self.review_text_proj(review_embedding_vectors)
        )

        retail_mask = (event_type_ids == EVENT_TYPE_RETAIL).unsqueeze(-1)
        review_mask = (event_type_ids == EVENT_TYPE_REVIEW).unsqueeze(-1)

        x = retail_mask * retail_vec + review_mask * review_vec
        x = x + self.event_type_emb(event_type_ids) + self.pos_emb(positions)
        x = self.encoder(x, src_key_padding_mask=~attention_mask)
        x = self.norm(x)

        last_idx = attention_mask.sum(dim=1) - 1
        last_hidden = x[torch.arange(batch_size, device=x.device), last_idx]
        logits = self.head(self.dropout(last_hidden))
        return logits


def hitrate_at_k(logits: torch.Tensor, targets: torch.Tensor, k: int = 10) -> float:
    topk = logits.topk(k=k, dim=1).indices
    hits = (topk == targets.unsqueeze(1)).any(dim=1).float()
    return hits.mean().item()


def mrr_at_k(logits: torch.Tensor, targets: torch.Tensor, k: int = 10) -> float:
    topk = logits.topk(k=k, dim=1).indices
    scores = []
    for pred, target in zip(topk, targets):
        positions = (pred == target).nonzero(as_tuple=False)
        scores.append(0.0 if len(positions) == 0 else 1.0 / (positions[0].item() + 1))
    return float(np.mean(scores))


def ndcg_at_k(logits: torch.Tensor, targets: torch.Tensor, k: int = 10) -> float:
    topk = logits.topk(k=k, dim=1).indices
    scores = []
    for pred, target in zip(topk, targets):
        positions = (pred == target).nonzero(as_tuple=False)
        scores.append(0.0 if len(positions) == 0 else 1.0 / math.log2(positions[0].item() + 2))
    return float(np.mean(scores))


In [101]:
def evaluate_model(model: nn.Module, loader: DataLoader, split_name: str = "eval") -> dict[str, float]:
    model.eval()
    all_logits = []
    all_targets = []
    print(f"Evaluating on {split_name}: {len(loader.dataset)} examples, {len(loader)} batches")
    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(loader, desc=f"eval:{split_name}"), start=1):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            logits = model(
                event_type_ids=batch["event_type_ids"],
                retail_item_ids=batch["retail_item_ids"],
                retail_action_ids=batch["retail_action_ids"],
                retail_subdomain_ids=batch["retail_subdomain_ids"],
                retail_os_ids=batch["retail_os_ids"],
                review_brand_ids=batch["review_brand_ids"],
                review_rating_ids=batch["review_rating_ids"],
                review_embedding_vectors=batch["review_embedding_vectors"],
                attention_mask=batch["attention_mask"],
            )
            all_logits.append(logits.cpu())
            all_targets.append(batch["targets"].cpu())

            if batch_idx % 20 == 0:
                print(f"eval {split_name}: processed {batch_idx} / {len(loader)} batches")

    logits = torch.cat(all_logits, dim=0)
    targets = torch.cat(all_targets, dim=0)
    return {
        "hr@10": hitrate_at_k(logits, targets, 10),
        "mrr@10": mrr_at_k(logits, targets, 10),
        "ndcg@10": ndcg_at_k(logits, targets, 10),
    }


def train_model(model: nn.Module, train_loader: DataLoader, valid_loader: DataLoader, epochs: int = EPOCHS):
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()
    best_state = None
    best_metric = -1.0
    history = []
    print(f"Start training: train={len(train_loader.dataset)} valid={len(valid_loader.dataset)}")

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        print(f"Epoch {epoch}/{epochs}: {len(train_loader)} train batches")

        for batch_idx, batch in enumerate(tqdm(train_loader, desc=f"train epoch {epoch}"), start=1):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optimizer.zero_grad()
            logits = model(
                event_type_ids=batch["event_type_ids"],
                retail_item_ids=batch["retail_item_ids"],
                retail_action_ids=batch["retail_action_ids"],
                retail_subdomain_ids=batch["retail_subdomain_ids"],
                retail_os_ids=batch["retail_os_ids"],
                review_brand_ids=batch["review_brand_ids"],
                review_rating_ids=batch["review_rating_ids"],
                review_embedding_vectors=batch["review_embedding_vectors"],
                attention_mask=batch["attention_mask"],
            )
            loss = criterion(logits, batch["targets"])
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(batch["targets"])
            if batch_idx % 20 == 0:
                avg_loss = running_loss / (batch_idx * train_loader.batch_size)
                print(f"epoch={epoch} batch={batch_idx}/{len(train_loader)} running_loss_per_example={avg_loss:.4f}")

        train_loss = running_loss / len(train_loader.dataset)
        print(f"Epoch {epoch} finished, train_loss={train_loss:.4f}. Start validation")
        valid_metrics = evaluate_model(model, valid_loader, split_name=f"valid@epoch{epoch}")
        history.append({"epoch": epoch, "train_loss": train_loss, **valid_metrics})
        print({"epoch": epoch, "train_loss": round(train_loss, 4), **{k: round(v, 4) for k, v in valid_metrics.items()}})

        if valid_metrics["ndcg@10"] > best_metric:
            best_metric = valid_metrics["ndcg@10"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    return pd.DataFrame(history)


## 10. Функция одного эксперимента


In [102]:
def run_experiment(history_domains: list[str], experiment_name: str) -> dict:
    train_raw, valid_raw, test_raw = build_examples(
        events=events,
        history_domains=history_domains,
        target_item_set=target_item_set,
        max_seq_len=MAX_SEQ_LEN,
    )

    vocabs = build_feature_vocabs(train_raw)
    train_encoded = encode_examples(train_raw, vocabs, split_name="train")
    valid_encoded = encode_examples(valid_raw, vocabs, split_name="valid")
    test_encoded = encode_examples(test_raw, vocabs, split_name="test")

    print(experiment_name)
    print("train / valid / test:", len(train_encoded), len(valid_encoded), len(test_encoded))
    print("target_vocab:", len(vocabs["target_item"]))

    train_loader = DataLoader(SeqDataset(train_encoded), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)
    valid_loader = DataLoader(SeqDataset(valid_encoded), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)
    test_loader = DataLoader(SeqDataset(test_encoded), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)

    model = HeterogeneousSeqTransformer(
        n_retail_items=len(vocabs["retail_item"]),
        n_retail_actions=len(vocabs["retail_action"]),
        n_retail_subdomains=len(vocabs["retail_subdomain"]),
        n_retail_os=len(vocabs["retail_os"]),
        n_review_brands=len(vocabs["review_brand"]),
        n_targets=len(vocabs["target_item"]),
    ).to(DEVICE)

    train_history = train_model(model, train_loader, valid_loader, epochs=EPOCHS)
    valid_metrics = evaluate_model(model, valid_loader, split_name="valid-final")
    test_metrics = evaluate_model(model, test_loader, split_name="test")

    return {
        "experiment": experiment_name,
        "domains": ", ".join(history_domains),
        "n_train": len(train_encoded),
        "n_valid": len(valid_encoded),
        "n_test": len(test_encoded),
        "retail_item_vocab": len(vocabs["retail_item"]),
        "review_brand_vocab": len(vocabs["review_brand"]),
        "target_vocab": len(vocabs["target_item"]),
        "valid_hr@10": valid_metrics["hr@10"],
        "valid_mrr@10": valid_metrics["mrr@10"],
        "valid_ndcg@10": valid_metrics["ndcg@10"],
        "test_hr@10": test_metrics["hr@10"],
        "test_mrr@10": test_metrics["mrr@10"],
        "test_ndcg@10": test_metrics["ndcg@10"],
        "history": train_history,
    }


## 11. Baseline и cross-domain experiment


In [103]:
experiments = [
    ("retail_only", ["retail"]),
    ("retail_plus_reviews", ["retail", "reviews"]),
]
experiments


[('retail_only', ['retail']), ('retail_plus_reviews', ['retail', 'reviews'])]

In [104]:
results = []
histories = {}

for experiment_name, history_domains in experiments:
    result = run_experiment(history_domains, experiment_name)
    histories[experiment_name] = result.pop("history")
    results.append(result)

results_df = pd.DataFrame(results)
results_df


Building examples for domains=['retail']
Total raw events: 50252


group events by user:   0%|          | 0/50252 [00:00<?, ?it/s]

Users with sequences: 700


build_examples:retail:   0%|          | 0/700 [00:00<?, ?it/s]

processed_users=500 train=10350 valid=449 test=449
Finished build_examples
train / valid / test: 14274 628 628
Encoding train examples: 14274


...

{'epoch': 4, 'train_loss': 6.3276, 'hr@10': 0.1066, 'mrr@10': 0.0522, 'ndcg@10': 0.0647}
Evaluating on valid-final: 610 examples, 3 batches


eval:valid-final:   0%|          | 0/3 [00:00<?, ?it/s]

Evaluating on test: 599 examples, 3 batches


eval:test:   0%|          | 0/3 [00:00<?, ?it/s]

,experiment,domains,n_train,n_valid,n_test,retail_item_vocab,review_brand_vocab,target_vocab,valid_hr@10,valid_mrr@10,valid_ndcg@10,test_hr@10,test_mrr@10,test_ndcg@10
0,retail_only,retail,14274,609,598,15684,2,2118,0.108374,0.054637,0.067166,0.095318,0.049218,0.059884
1,retail_plus_reviews,"retail, reviews",14318,610,599,15684,258,2121,0.106557,0.052165,0.064654,0.100167,0.046250,0.058775


In [105]:
metric_cols = [
    "valid_hr@10", "valid_mrr@10", "valid_ndcg@10",
    "test_hr@10", "test_mrr@10", "test_ndcg@10",
]

results_df[metric_cols] = results_df[metric_cols].round(4)
results_df.sort_values("test_ndcg@10", ascending=False)


,experiment,domains,n_train,n_valid,n_test,retail_item_vocab,review_brand_vocab,target_vocab,valid_hr@10,valid_mrr@10,valid_ndcg@10,test_hr@10,test_mrr@10,test_ndcg@10
0,retail_only,retail,14274,609,598,15684,2,2118,0.1084,0.0546,0.0672,0.0953,0.0492,0.0599
1,retail_plus_reviews,"retail, reviews",14318,610,599,15684,258,2121,0.1066,0.0522,0.0647,0.1002,0.0462,0.0588


## 12. Что анализировать

1. Есть ли прирост `NDCG@10` при добавлении reviews в историю.
2. Не слишком ли выросла сложность модели ради малого прироста.
3. Помогает ли reviews-сигнал только части пользователей.
4. Что лучше работает для review encoder-а: только `brand_id + rating` или еще и dense `embedding`.
